# Scrub Typhus Transmission Dynamics: Human–Mite–Rodent Model

**Computational notebook for symbolic analysis, reproduction-number calculation, and sensitivity analysis**

This notebook organises the Scrub Typhus modelling work into a reproducible sequence:

1. Define the ten-compartment Human–Mite–Rodent system.
2. Construct the full-system Jacobian and evaluate it at the disease-free equilibrium (DFE).
3. Form the next-generation matrix (NGM) for the infected subsystem.
4. Derive and evaluate the basic reproduction number, \(R_0\).
5. Calculate normalized local sensitivity indices.
6. Perform one-at-a-time parameter variation.
7. Perform Latin hypercube sampling (LHS) and partial rank correlation coefficient (PRCC) analysis.

> **Important model-review note:** The source notebook contained several unrelated/duplicate model versions and inconsistent treatment notation between the full ODE system and the NGM derivation. This cleaned version uses the ten-compartment ODE system as the reference and aligns the NGM removal terms with its single treatment parameter `tau`. This alignment should be checked against the final manuscript equations before publication. Parameter values and sampling ranges are retained from the source notebook where possible; they are not presented as independently validated literature estimates.

## 1. Model structure and notation

The state vector is

\[
X=(S_H,E_H,I_H,R_H,S_{QM},I_{FM},I_{AM},I_{QM},S_R,I_R)^T.
\]

| Host/vector group | Compartments | Description |
|---|---|---|
| Humans | \(S_H,E_H,I_H,R_H\) | Susceptible, exposed, infectious, recovered |
| Mites | \(S_{QM},I_{FM},I_{AM},I_{QM}\) | Susceptible questing mites and infected feeding, adult, and questing mites |
| Rodents | \(S_R,I_R\) | Susceptible and infected rodents |

The model uses `c1` for the mite-to-human contact coefficient and `c2` in the mite–rodent transmission terms, following the notation in the source work. Environmental modifiers are represented by temperature \(T\) and rainfall index \(r\).

**Interpretation to verify:** The source notebook uses `G` as a multiplier in mite-stage transitions. Its comments alternately describe `G` as a transovarial transmission parameter and as a mite growth-related parameter. The biological interpretation and units of `G` must be made consistent in the manuscript and parameter table.

## 2. Import required libraries

The notebook uses standard scientific Python packages. No package installation command is required in a normal Colab or Jupyter environment.

In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt

from scipy.stats import qmc, rankdata, pearsonr
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 110
})

pd.set_option("display.precision", 6)


## 3. Define the full ten-compartment model

The equations below reproduce the full system from the source notebook's ten-compartment model cell. The equations are retained rather than replacing the model with a different formulation.

In [ ]:
# State variables
S_H, E_H, I_H, R_H = sp.symbols("S_H E_H I_H R_H")
S_QM, I_FM, I_AM, I_QM = sp.symbols("S_QM I_FM I_AM I_QM")
S_R, I_R = sp.symbols("S_R I_R")

X = sp.Matrix([
    S_H, E_H, I_H, R_H,
    S_QM, I_FM, I_AM, I_QM,
    S_R, I_R
])

# Model parameters
b_H, b_M, b_R = sp.symbols("b_H b_M b_R", positive=True)
c1, c2 = sp.symbols("c1 c2", positive=True)
beta_H, beta_M, beta_R = sp.symbols("beta_H beta_M beta_R", positive=True)
alpha = sp.symbols("alpha", nonnegative=True)
rho_T, T = sp.symbols("rho_T T", nonnegative=True)
rho_r, r = sp.symbols("rho_r r", nonnegative=True)
G = sp.symbols("G", positive=True)

epsilon_H, delta_H = sp.symbols("epsilon_H delta_H", positive=True)
mu_H, sigma_H, gamma_H, tau = sp.symbols(
    "mu_H sigma_H gamma_H tau", positive=True
)
mu_M, omega, eta_M, theta_M = sp.symbols(
    "mu_M omega eta_M theta_M", positive=True
)
mu_R, phi = sp.symbols("mu_R phi", positive=True)

# Humans
f1 = b_H - c1*beta_H*(1-alpha)*(1+rho_T*T)*I_QM*S_H - mu_H*S_H - alpha*S_H
f2 = c1*beta_H*(1-alpha)*(1+rho_T*T)*I_QM*S_H - (epsilon_H + delta_H + mu_H)*E_H
f3 = delta_H*E_H - (mu_H + sigma_H + gamma_H + tau)*I_H
f4 = epsilon_H*E_H + (gamma_H + tau)*I_H - mu_H*R_H

# Mites
f5 = b_M - c2*beta_M*I_R*S_QM*G*(1+rho_r*r) - (mu_M + omega)*S_QM
f6 = c2*beta_M*I_R*S_QM*G*(1+rho_r*r) - (mu_M + omega)*I_FM - eta_M*G*(1+rho_r*r)*I_FM
f7 = eta_M*G*(1+rho_r*r)*I_FM - (mu_M + omega)*I_AM - theta_M*G*(1+rho_r*r)*I_AM
f8 = theta_M*G*(1+rho_r*r)*I_AM - (mu_M + omega)*I_QM

# Rodents
f9 = b_R - c2*beta_R*(1+rho_T*T)*I_QM*S_R - (mu_R + phi)*S_R
f10 = c2*beta_R*(1+rho_T*T)*I_QM*S_R - (mu_R + phi)*I_R

F_full = sp.Matrix([f1, f2, f3, f4, f5, f6, f7, f8, f9, f10])

display(pd.DataFrame({
    "Compartment": [str(v) for v in X],
    "Differential equation": [sp.Eq(sp.Symbol(f"d{v}/dt"), rhs) for v, rhs in zip(X, F_full)]
}))


## 4. Disease-free equilibrium and full-system Jacobian

At the DFE, all exposed and infected compartments are zero. The susceptible equilibrium values follow from the recruitment and removal terms in the stated ODE system.

In [ ]:
# Disease-free equilibrium
DFE = {
    E_H: 0, I_H: 0, R_H: 0,
    I_FM: 0, I_AM: 0, I_QM: 0, I_R: 0,
    S_H: b_H/(mu_H + alpha),
    S_QM: b_M/(mu_M + omega),
    S_R: b_R/(mu_R + phi)
}

J = F_full.jacobian(X)
J_DFE = sp.simplify(J.subs(DFE))

print("Disease-free equilibrium:")
display(DFE)

print("Full-system Jacobian evaluated at the DFE:")
display(J_DFE)


## 5. Next-generation matrix

The infected-state vector is

\[
Y=(E_H,I_H,I_{FM},I_{AM},I_{QM},I_R)^T.
\]

The new-infection vector \(\mathcal{F}\) and transition/removal vector \(\mathcal{V}\) are constructed from the full-system equations. In particular, the NGM uses the same single treatment parameter `tau` as the full ODE system; the source notebook's separate `tau1`, `tau2`, and `lambda_H` notation was not consistent with that system.

The susceptible quantities in the NGM are evaluated at the DFE. For the numerical calculation below, the supplied susceptible population values are used directly as DFE values.

In [ ]:
# Infected variables
E_H_i, I_H_i, I_FM_i, I_AM_i, I_QM_i, I_R_i = sp.symbols(
    "E_H_i I_H_i I_FM_i I_AM_i I_QM_i I_R_i"
)
Y = sp.Matrix([E_H_i, I_H_i, I_FM_i, I_AM_i, I_QM_i, I_R_i])

# New infection terms
Lambda_H = c1*beta_H*(1-alpha)*(1+rho_T*T)*I_QM_i*S_H
Lambda_M = c2*beta_M*I_R_i*S_QM*G*(1+rho_r*r)
Lambda_R = c2*beta_R*(1+rho_T*T)*I_QM_i*S_R

F_terms = sp.Matrix([Lambda_H, 0, Lambda_M, 0, 0, Lambda_R])

# Remaining transitions and removals
V_terms = sp.Matrix([
    (epsilon_H + delta_H + mu_H)*E_H_i,
    (mu_H + sigma_H + gamma_H + tau)*I_H_i - delta_H*E_H_i,
    (mu_M + omega + eta_M*G*(1+rho_r*r))*I_FM_i,
    (mu_M + omega + theta_M*G*(1+rho_r*r))*I_AM_i
        - eta_M*G*(1+rho_r*r)*I_FM_i,
    (mu_M + omega)*I_QM_i - theta_M*G*(1+rho_r*r)*I_AM_i,
    (mu_R + phi)*I_R_i
])

F_matrix = F_terms.jacobian(Y)
V_matrix = V_terms.jacobian(Y)

# Evaluate at disease-free state (infected variables are already linearised)
F_DFE = sp.simplify(F_matrix)
V_DFE = sp.simplify(V_matrix)

NGM = sp.simplify(F_DFE * V_DFE.inv())

print("F matrix:")
display(F_DFE)
print("V matrix:")
display(V_DFE)
print("Next-generation matrix (F V^{-1}):")
display(NGM)


## 6. Basic reproduction number, \(R_0\)

The non-zero eigenvalues of the NGM determine its spectral radius. The expression below is the closed-form expression encoded in the source notebook. The following cell verifies the characteristic polynomial structure symbolically.

In [ ]:
# Characteristic polynomial of the NGM
z = sp.symbols("z")
characteristic_polynomial = sp.factor(NGM.charpoly(z).as_expr())

# Closed-form R0 expression from the source derivation
R0_expression = c2 * sp.sqrt(
    (
        G**3 * S_QM * S_R * beta_M * beta_R * eta_M * theta_M
        * (1 + rho_T*T) * (1 + rho_r*r)**3
    ) / (
        (mu_M + omega) * (mu_R + phi)
        * (G*eta_M*(1 + rho_r*r) + mu_M + omega)
        * (G*theta_M*(1 + rho_r*r) + mu_M + omega)
    )
)

print("Characteristic polynomial:")
display(characteristic_polynomial)

print("Closed-form R0:")
display(sp.Eq(sp.Symbol("R_0"), R0_expression))

# R0 does not contain beta_H or c1 because humans are modelled as a dead-end host:
# the human infection pathway does not return infection to mites or rodents.


## 7. Numerical parameter set and baseline \(R_0\)

The baseline values below are taken from the later parameter set in the source notebook (including `G = 0.084`, temperature `T = 28`, and rainfall index `r = 110`). They should be replaced only after checking the corresponding units, sources, and study-area data.

The calculated \(R_0\) is a model output for this parameter set; it is not independent validation of the parameter values.

In [ ]:
parameter_values = {
    S_QM: 120000,       # Susceptible questing mites
    S_R: 35000,         # Susceptible rodents
    beta_M: 2.5e-6,     # Mite/rodent transmission coefficient as coded
    beta_R: 1.8e-6,     # Rodent/mite transmission coefficient as coded
    eta_M: 0.18,        # Mite-stage progression parameter
    theta_M: 0.12,      # Mite-stage progression parameter
    T: 28,              # Mean temperature (unit must be confirmed)
    r: 110,             # Rainfall index (definition/unit must be confirmed)
    rho_T: 0.015,       # Temperature effect coefficient
    rho_r: 0.005,       # Rainfall effect coefficient
    mu_M: 0.03,         # Mite mortality/removal rate
    mu_R: 0.015,        # Rodent mortality/removal rate
    omega: 0.01,        # Mite control/removal parameter
    phi: 0.008,         # Rodent control/removal parameter
    G: 0.084,           # Meaning and units require confirmation
    c2: 0.65            # Mite-rodent contact coefficient
}

R0_numeric = float(sp.N(R0_expression.subs(parameter_values)))
print(f"Baseline R0 = {R0_numeric:.6f}")

parameter_table = pd.DataFrame({
    "Parameter": [str(k) for k in parameter_values],
    "Value": list(parameter_values.values())
})
display(parameter_table)


## 8. Normalized local sensitivity analysis

For a parameter \(p\), the normalized sensitivity index is

\[
\Upsilon_p=\frac{\partial R_0}{\partial p}\frac{p}{R_0}.
\]

A positive index indicates that increasing the parameter increases \(R_0\) locally; a negative index indicates the opposite. The magnitude represents the relative local effect.

In [ ]:
sensitivity_results = {}

for parameter in parameter_values:
    derivative = sp.diff(R0_expression, parameter)
    index_expression = sp.simplify((parameter / R0_expression) * derivative)
    sensitivity_results[str(parameter)] = float(
        sp.N(index_expression.subs(parameter_values))
    )

sensitivity_df = (
    pd.DataFrame(
        sensitivity_results.items(),
        columns=["Parameter", "Normalized sensitivity index"]
    )
    .assign(Absolute_index=lambda df: df["Normalized sensitivity index"].abs())
    .sort_values("Absolute_index", ascending=False)
    .drop(columns="Absolute_index")
    .reset_index(drop=True)
)

display(sensitivity_df)

plot_df = sensitivity_df.sort_values("Normalized sensitivity index")
plt.figure(figsize=(10, 6))
plt.barh(
    plot_df["Parameter"],
    plot_df["Normalized sensitivity index"],
    edgecolor="black"
)
plt.axvline(0, linewidth=1)
plt.xlabel("Normalized sensitivity index")
plt.ylabel("Parameter")
plt.title("Local sensitivity of R₀")
plt.tight_layout()
plt.show()


## 9. One-at-a-time parameter variation

Each parameter is varied independently by ±30% around its baseline value while all other parameters are held fixed. This is a local exploratory analysis, not a substitute for uncertainty analysis or parameter estimation.

In [ ]:
relative_change = np.linspace(-30, 30, 121)

# Symbols in the R0 expression that have baseline values
sweep_parameters = list(parameter_values.keys())

fig, axes = plt.subplots(4, 4, figsize=(15, 12))
axes = axes.ravel()

for ax, parameter in zip(axes, sweep_parameters):
    baseline = parameter_values[parameter]
    r0_values = []

    for percent in relative_change:
        varied = dict(parameter_values)
        varied[parameter] = baseline * (1 + percent / 100)
        r0_values.append(float(sp.N(R0_expression.subs(varied))))

    ax.plot(relative_change, r0_values, linewidth=2)
    ax.axvline(0, linewidth=0.8)
    ax.axhline(R0_numeric, linestyle="--", linewidth=0.8)
    ax.set_title(str(parameter))
    ax.set_xlabel("Change from baseline (%)")
    ax.set_ylabel("R₀")

for ax in axes[len(sweep_parameters):]:
    ax.set_visible(False)

fig.suptitle("One-at-a-time sensitivity of R₀", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()


## 10. Global sensitivity analysis: LHS + PRCC

Latin hypercube sampling is used to explore the parameter ranges specified in the source notebook. PRCC is computed by rank-transforming the variables and correlating residuals after removing the linear effects of the remaining ranked parameters.

**Important:** The ranges below are carried over from the source notebook and are exploratory, not confirmed literature ranges. In particular, the source baseline has `G = 0.084`, whereas the original LHS range was `0.30–0.95`. This means the baseline is outside the sampled range. Resolve this inconsistency before interpreting or publishing the PRCC results.

In [ ]:
def compute_R0_from_dict(p):
    numerator = (
        p["G"]**3 * p["S_QM"] * p["S_R"]
        * p["beta_M"] * p["beta_R"]
        * p["eta_M"] * p["theta_M"]
        * (1 + p["rho_T"]*p["T"])
        * (1 + p["rho_r"]*p["r"])**3
    )
    denominator = (
        (p["mu_M"] + p["omega"])
        * (p["mu_R"] + p["phi"])
        * (p["G"]*p["eta_M"]*(1 + p["rho_r"]*p["r"]) + p["mu_M"] + p["omega"])
        * (p["G"]*p["theta_M"]*(1 + p["rho_r"]*p["r"]) + p["mu_M"] + p["omega"])
    )
    return p["c2"] * np.sqrt(numerator / denominator)


parameter_names = [
    "S_QM", "S_R", "beta_M", "beta_R", "eta_M", "theta_M",
    "T", "r", "rho_T", "rho_r", "mu_M", "mu_R",
    "omega", "phi", "G", "c2"
]

# Exploratory ranges retained from the source notebook
parameter_bounds = {
    "S_QM": (50000, 200000),
    "S_R": (10000, 70000),
    "beta_M": (1e-7, 5e-6),
    "beta_R": (1e-7, 5e-6),
    "eta_M": (0.05, 0.40),
    "theta_M": (0.05, 0.35),
    "T": (18, 35),
    "r": (50, 180),
    "rho_T": (0.005, 0.03),
    "rho_r": (0.001, 0.01),
    "mu_M": (0.005, 0.08),
    "mu_R": (0.005, 0.05),
    "omega": (0.001, 0.05),
    "phi": (0.001, 0.04),
    "G": (0.30, 0.95),
    "c2": (0.20, 1.20)
}

n_samples = 1000
sampler = qmc.LatinHypercube(d=len(parameter_names), seed=42)
unit_sample = sampler.random(n=n_samples)

lower = np.array([parameter_bounds[name][0] for name in parameter_names])
upper = np.array([parameter_bounds[name][1] for name in parameter_names])
sample_values = qmc.scale(unit_sample, lower, upper)

lhs_df = pd.DataFrame(sample_values, columns=parameter_names)
lhs_df["R0"] = lhs_df.apply(lambda row: compute_R0_from_dict(row.to_dict()), axis=1)

print(f"Number of LHS simulations: {len(lhs_df)}")
print(lhs_df["R0"].describe())
display(lhs_df.head())


In [ ]:
def calculate_prcc(data, parameter_columns, outcome_column):
    """Calculate PRCC using rank transformation and residual correlation."""
    ranked = data[parameter_columns + [outcome_column]].rank(method="average")
    y = ranked[outcome_column].to_numpy(dtype=float)
    results = []

    for parameter in parameter_columns:
        x = ranked[parameter].to_numpy(dtype=float)
        covariates = [p for p in parameter_columns if p != parameter]

        if covariates:
            Z = ranked[covariates].to_numpy(dtype=float)
            Z = np.column_stack([np.ones(len(Z)), Z])
            beta_x = np.linalg.lstsq(Z, x, rcond=None)[0]
            beta_y = np.linalg.lstsq(Z, y, rcond=None)[0]
            residual_x = x - Z @ beta_x
            residual_y = y - Z @ beta_y
        else:
            residual_x, residual_y = x, y

        r_value, p_value = pearsonr(residual_x, residual_y)
        results.append((parameter, r_value, p_value))

    return (
        pd.DataFrame(results, columns=["Parameter", "PRCC", "p-value"])
        .sort_values("PRCC", ascending=False)
        .reset_index(drop=True)
    )


prcc_results = calculate_prcc(lhs_df, parameter_names, "R0")
display(prcc_results)

plt.figure(figsize=(10, 7))
ordered = prcc_results.sort_values("PRCC")
plt.barh(ordered["Parameter"], ordered["PRCC"], edgecolor="black")
plt.axvline(0, linewidth=1)
plt.xlabel("Partial rank correlation coefficient (PRCC)")
plt.ylabel("Parameter")
plt.title("Global sensitivity analysis of R₀ (LHS–PRCC)")
plt.tight_layout()
plt.show()

print("Parameters with p-value < 0.05:")
display(prcc_results.loc[prcc_results["p-value"] < 0.05])


## 11. Interpretation and checks before publication

- Positive local sensitivity/PRCC values indicate a positive association with \(R_0\); negative values indicate a negative association within the analysed parameter ranges.
- The model structure makes humans a dead-end host: human infection does not feed back into mite or rodent infection. Therefore, the human-to-mite contact coefficient `c1` and human transmission coefficient `beta_H` do not appear in the closed-form \(R_0\) expression for the mite–rodent transmission cycle.
- The numerical baseline \(R_0\) depends on the supplied parameter values and units. An \(R_0\) value below one should not be achieved by arbitrary parameter tuning; parameters must be justified independently.
- Confirm the biological meaning and units of `G`, `r`, `T`, `beta_M`, `beta_R`, `eta_M`, and `theta_M`.
- Confirm whether `omega` and `phi` are rates and whether their effects are represented consistently in all compartments.
- Reconcile the treatment/awareness terms in the manuscript with the full ODE system. This notebook uses `tau` consistently in the NGM to match the full-system cell.
- Validate the parameter table and environmental inputs against the intended study location and period before making epidemiological claims.
- The notebook does not include empirical data fitting or independent model validation. Those analyses require the underlying surveillance and environmental datasets.

## 12. Reproducibility

The notebook is designed to run from top to bottom in Google Colab or a local Jupyter environment. The random seed for LHS is fixed (`42`) so the sampling design is reproducible. Numerical results can still vary if package versions or parameter inputs change.